# 🎙️ Speech Intelligence Engine — Google Colab GPU Trainer

This notebook offloads heavy **Speech Foundation Model** training, SSL feature extraction (WavLM / HuBERT), and Speech-LLM adapter fine-tuning to Google Colab GPUs (T4 / L4 / A100), persisting all checkpoints directly to your 5TB **Google Drive** (`rakshithgoud453@gmail.com`).

## Step 1: Check GPU & Mount Google Drive (5TB Storage)

In [12]:
# Check available GPU resources
!nvidia-smi

# Mount 5TB Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
DATASET_DIR = os.path.join(DRIVE_BASE_PATH, 'datasets')
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, 'outputs')

for path in [DRIVE_BASE_PATH, CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)
    print(f"✓ Ready directory on Google Drive: {path}")

Fri Oct  2 08:02:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             28W /   70W |     953MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Install ML & Audio Dependencies (Colab-Optimized)

In [13]:
# Install NLP, Audio & ML packages with numpy<2.0.0 pin
!pip install -q "numpy<2.0.0" transformers datasets peft accelerate bitsandbytes librosa soundfile wandb faster-whisper hdbscan


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyannote-metrics 4.1 requires numpy>=2.2.2, but you have numpy 1.26.4 which is incompatible.
pyannote-core 6.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.9.15 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
dm-tree 0.1.10 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.11.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which

## Step 3: Clone / Sync Repository Code

In [14]:
# Codebase setup in Colab workspace
import os
import sys
import importlib

REPO_URL = "https://github.com/rakshithgoud453/speech_intelligence_engine.git"
WORKSPACE_DIR = "/content/speech_intelligence_engine"

%cd /content
if os.path.exists(WORKSPACE_DIR):
    %cd {WORKSPACE_DIR}
    !git pull
else:
    !git clone {REPO_URL} {WORKSPACE_DIR}
    %cd {WORKSPACE_DIR}

# Purge cached modules so git pull updates take effect immediately
for mod in list(sys.modules.keys()):
    if mod.startswith("speech_intelligence_engine") or mod in ["engine", "prosody"]:
        sys.modules.pop(mod, None)

importlib.invalidate_caches()
print(f"✓ Currently operating in: {os.getcwd()}")


/content
/content/speech_intelligence_engine
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 9 (delta 6), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 12.61 KiB | 192.00 KiB/s, done.
From https://github.com/rakshithgoud453/speech_intelligence_engine
   119835f..aa3f172  master     -> origin/master
Updating 119835f..aa3f172
Fast-forward
 engine.py                                         |  38 ++--
 notebooks/speech_intelligence_colab_trainer.ipynb | 214 ++++++++++++++++------
 2 files changed, 187 insertions(+), 65 deletions(-)
✓ Currently operating in: /content/speech_intelligence_engine


## Step 4: SSL Layer Representation Extraction (WavLM / HuBERT)

In [15]:
import torch
import torchaudio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Pre-trained SSL Model natively via Torchaudio
bundle = torchaudio.pipelines.WAVLM_BASE
ssl_model = bundle.get_model().to(device)
ssl_model.eval()

def extract_ssl_representations(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    if sample_rate != bundle.sample_rate:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=bundle.sample_rate)
        waveform = resampler(waveform)

    with torch.no_grad():
        all_layers, _ = ssl_model.extract_features(waveform.to(device))

    # Layers 4-8: Prosodic/Phonetic representations
    prosodic_rep = torch.stack(all_layers[4:9]).mean(dim=0).squeeze(0)
    # Layers 9-11: Semantic representations
    semantic_rep = torch.stack(all_layers[9:12]).mean(dim=0).squeeze(0)

    return {
        "prosodic_embedding": prosodic_rep.cpu().numpy(),
        "semantic_embedding": semantic_rep.cpu().numpy()
    }

print("✓ SSL Feature Extraction Pipeline Loaded Successfully.")


Using device: cuda
✓ SSL Feature Extraction Pipeline Loaded Successfully.


## Step 5: Checkpoint Saver Utility

In [16]:
import os
import time
import torch

DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(model, step, metrics):
    checkpoint_name = f"speech_engine_step_{step}_{int(time.time())}.pt"
    save_path = os.path.join(CHECKPOINT_DIR, checkpoint_name)

    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'metrics': metrics
    }, save_path)

    print(f"✓ Saved Checkpoint to Google Drive (5TB): {save_path}")

print(f"✓ Checkpoints directory configured: {CHECKPOINT_DIR}")

✓ Checkpoints directory configured: /content/drive/MyDrive/speech_intelligence_engine/checkpoints


## Step 6: Batch Audio Ingestion & Model Checkpoint Execution Loop 🚀 (Self-Contained & Auto-Logging)

In [17]:
import importlib
for mod in list(sys.modules.keys()):
    if mod.startswith("speech_intelligence_engine") or mod in ["engine", "prosody"]:
        sys.modules.pop(mod, None)
importlib.invalidate_caches()
import sys
import glob
import json
import os
import time
import traceback
import soundfile as sf
import numpy as np
import torch

# 1. Ensure Google Drive Output Directories
DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
DATASET_DIR = os.path.join(DRIVE_BASE_PATH, 'datasets')
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, 'outputs')
for p in [CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR]:
    os.makedirs(p, exist_ok=True)

# 2. Local Saver Fallback
def local_save_checkpoint(model, step, metrics):
    checkpoint_name = f"speech_engine_step_{step}_{int(time.time())}.pt"
    save_path = os.path.join(CHECKPOINT_DIR, checkpoint_name)
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'metrics': metrics
    }, save_path)
    print(f"✓ Saved Checkpoint to 5TB Google Drive: {save_path}")

# 3. Execute Loop with Automated Error Logging
try:
    WORKSPACE_DIR = "/content/speech_intelligence_engine"
    for p in [WORKSPACE_DIR, os.path.dirname(WORKSPACE_DIR)]:
        if p not in sys.path:
            sys.path.insert(0, p)

    try:
        from speech_intelligence_engine import SpeechLearningEngine
    except ImportError:
        from engine import SpeechLearningEngine

    try:
        from speech_intelligence_engine.representation.projector import SpeechLLMProjector
    except ImportError:
        from representation.projector import SpeechLLMProjector

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Initializing Engine & Projector on device: {device}...")

    engine = SpeechLearningEngine(model_type="wavlm", replay_capacity=1000)
    projector = SpeechLLMProjector(input_dim=768, hidden_dim=2048, output_dim=4096).to(device)

    audio_files = glob.glob(os.path.join(DATASET_DIR, "*.wav")) + glob.glob(os.path.join(DATASET_DIR, "*.mp3"))

    if not audio_files:
        print(f"No audio files in Google Drive datasets. Generating benchmark audio sample...")
        sample_wav = os.path.join(DATASET_DIR, "sample_speech_benchmark.wav")
        t = np.linspace(0, 3.0, int(16000 * 3.0), endpoint=False)
        sig = (0.5 * np.sin(2 * np.pi * 180 * t) + 0.2 * np.sin(2 * np.pi * 360 * t)).astype(np.float32)
        sf.write(sample_wav, sig, 16000)
        audio_files = [sample_wav]

    for i, audio_path in enumerate(audio_files, 1):
        print(f"\n[Processing {i}/{len(audio_files)}]: {audio_path}")
        res = engine.assimilate(audio_path, recording_id=os.path.basename(audio_path))

        metrics = {
            "episodes": res.num_episodes,
            "voiced_segments": res.num_segments,
            "novelty_rate": res.novelty_rate,
            "duration_sec": res.duration_sec
        }
        local_save_checkpoint(projector, step=i, metrics=metrics)

    status_payload = {
        "status": "COMPLETED",
        "total_episodes_learned": engine.get_state().total_episodes,
        "prototypes_discovered": engine.get_state().total_prototypes,
        "checkpoint_directory": CHECKPOINT_DIR,
        "last_updated": time.ctime()
    }

    status_file = os.path.join(OUTPUT_DIR, "status.json")
    with open(status_file, "w") as f:
        json.dump(status_payload, f, indent=2)

    print(f"\n✨ Batch processing complete!")
    print(f"📁 Checkpoint saved to Google Drive: {CHECKPOINT_DIR}")
    print(f"📄 Status summary written to: {status_file}")

except Exception as e:
    err_trace = traceback.format_exc()
    print(f"\n❌ EXECUTION ERROR IN STEP 6:\n{err_trace}")
    error_log_path = os.path.join(OUTPUT_DIR, "error_log.txt")
    with open(error_log_path, "w") as f:
        f.write(err_trace)
    print(f"📄 Full Error Traceback written to Google Drive: {error_log_path}")
    raise e

🚀 Initializing Engine & Projector on device: cuda...

[Processing 1/4]: /content/drive/MyDrive/speech_intelligence_engine/datasets/LEARN ENGLISH While Exploring Nature ｜ Comprehensible Input.wav

❌ EXECUTION ERROR IN STEP 6:
Traceback (most recent call last):
  File "/tmp/ipykernel_1920/60139493.py", line 65, in <cell line: 0>
    res = engine.assimilate(audio_path, recording_id=os.path.basename(audio_path))
  File "/content/speech_intelligence_engine/engine.py", line 170, in assimilate
    waveform, sample_rate = self._load_audio(audio_input, sr)
                            ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/content/speech_intelligence_engine/engine.py", line 427, in _load_audio
    try:
    
  File "/usr/local/lib/python3.13/dist-packages/lazy_loader/__init__.py", line 79, in __getattr__
    attr = getattr(submod, name)
  File "/usr/local/lib/python3.13/dist-packages/lazy_loader/__init__.py", line 78, in __getattr__
    submod = importlib.import_module(submod_path)
  File "/u

## Step 7: Native Speech-LLM LoRA Adapter Alignment & Multimodal Inference 🧠

In this step, raw WavLM 768-d continuous acoustic/prosodic latents are projected directly into LLM token space (), allowing a lightweight LoRA-adapted LLM backbone to reason over speech *without* cascaded text transcription.

In [ ]:
import os
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from speech_intelligence_engine.representation.projector import SpeechLLMProjector

DRIVE_BASE_PATH = "/content/drive/MyDrive/speech_intelligence_engine"
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, "checkpoints")
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, "outputs")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LLM_MODEL_ID = "Qwen/Qwen1.5-0.5B"
print(f"🚀 Initializing Speech-LLM Multimodal Adapter ({LLM_MODEL_ID})...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
llm = get_peft_model(llm, lora_config)
llm.print_trainable_parameters()

projector_dim = llm.config.hidden_size
projector = SpeechLLMProjector(input_dim=768, hidden_dim=2048, output_dim=projector_dim).to(device)

speech_latents = torch.randn(1, 50, 768, device=device)
projected_tokens = projector(speech_latents)

llm_embeddings = llm.get_input_embeddings()
text_input = "Analyze spoken input:"
text_tokens = tokenizer(text_input, return_tensors="pt").input_ids.to(device)
text_embeds = llm_embeddings(text_tokens)

multimodal_embeds = torch.cat([projected_tokens.to(text_embeds.dtype), text_embeds], dim=1)

with torch.no_grad():
    output_ids = llm.generate(inputs_embeds=multimodal_embeds, max_new_tokens=30)
    response_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Speech-LLM Multimodal Forward Pass Verified!")
print(f"🎙️ Speech Latent Tokens: {projected_tokens.shape}")
print(f"💬 Native LLM Multimodal Response: '{response_text.strip()}'")

adapter_save_path = os.path.join(CHECKPOINT_DIR, "speech_llm_lora_adapter")
llm.save_pretrained(adapter_save_path)
torch.save(projector.state_dict(), os.path.join(adapter_save_path, "speech_projector.pt"))
print(f"✓ Saved Speech-LLM LoRA Adapter & Projector to Google Drive: {adapter_save_path}")
